# 💰 Meal Profitability Clustering
### Gaussian Mixture Model (GMM)

**Course:** Machine Learning  
**Approach:** Unsupervised Learning  

---

## What are we trying to do?

We want to group meals into profitability clusters:
- 🟢 **HIGH profit** meals — keep and promote
- 🟡 **MEDIUM profit** meals — optimise
- 🔴 **LOW profit** meals — reprice or remove

We also want to see how each meal's profitability **changes across academic periods** 
(Normal Week, Exam Week, Orientation Week, Holiday).

---

## Why GMM and not K-Means?

K-Means forces every meal into exactly one cluster with 100% certainty.
But in reality, some meals are borderline — profitable on some days and not others.

GMM gives each meal a **probability score** instead:

```
Rice & Stew:  HIGH profit → 72%   MEDIUM → 24%   LOW → 4%
Fresh Juice:  HIGH profit → 41%   MEDIUM → 51%   LOW → 8%  ← borderline meal
```

This is more realistic and helps managers make better decisions.

---
## Step 1 — Import Libraries
These are all the tools we need for this notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
%matplotlib inline
np.random.seed(42)

print('✅ All libraries imported')

---
## Step 2 — Load the Dataset
> Replace `'your_cafeteria_data.csv'` with your actual file name.

In [ ]:
df = pd.read_csv('your_cafeteria_data.csv', parse_dates=['Date'])

print('Rows    :', len(df))
print('Columns :', df.columns.tolist())
print('Meals   :', df['Meal'].unique())
print('Periods :', df['Academic_Period'].unique())
print()
df.head(5)

---
## Step 3 — Compute Profitability Features
We calculate four simple features that describe how profitable each meal is:

| Feature | What it means |
|---|---|
| `Profit_Margin` | Out of every 1 UGX earned, how much is actual profit? |
| `Waste_Ratio` | Out of every 1 UGX earned, how much was wasted? |
| `Sell_Through` | Out of every portion prepared, how many actually sold? |
| `Revenue_Per_Sold` | How much does each sold portion earn? |

In [ ]:
df = df.sort_values(['Meal', 'Date']).reset_index(drop=True)

# These four features capture profitability from different angles
df['Profit_Margin']    = df['Gross_Profit_UGX']    / df['Revenue_UGX'].replace(0, np.nan)
df['Waste_Ratio']      = df['Waste_Cost_UGX']      / df['Revenue_UGX'].replace(0, np.nan)
df['Sell_Through']     = df['Portions_Sold']        / df['Portions_Prepared'].replace(0, np.nan)
df['Revenue_Per_Sold'] = df['Revenue_UGX']          / df['Portions_Sold'].replace(0, np.nan)

# Remove any rows with missing values
df = df.dropna(subset=['Profit_Margin','Waste_Ratio','Sell_Through','Revenue_Per_Sold'])
df = df.reset_index(drop=True)

print(f'Clean records: {len(df)}')
print()

# Quick summary: average profitability per meal
df.groupby('Meal')[['Profit_Margin','Waste_Ratio','Sell_Through']].mean().round(3)

---
## Step 4 — Aggregate by Meal and Academic Period
Instead of clustering individual daily records, we summarise each 
**(Meal × Academic Period)** combination into one row.

This means the GMM will answer: *'Is Rice & Stew during Exam Week a profitable combination?'*

In [ ]:
agg = (
    df.groupby(['Cafeteria_ID', 'Meal', 'Academic_Period'])
    .agg(
        Profit_Margin    = ('Profit_Margin',    'mean'),
        Waste_Ratio      = ('Waste_Ratio',      'mean'),
        Sell_Through     = ('Sell_Through',     'mean'),
        Revenue_Per_Sold = ('Revenue_Per_Sold', 'mean'),
        Avg_Gross_Profit = ('Gross_Profit_UGX', 'mean'),
        Avg_Revenue      = ('Revenue_UGX',      'mean'),
    )
    .reset_index()
)

print(f'Total combinations (meal × period): {len(agg)}')
print()
agg.head(10)

---
## Step 5 — Quick Look at Profitability
Before clustering, let us visually explore which meals and periods look more profitable.

In [ ]:
# Profit margin per meal per academic period — heatmap
pivot = agg.pivot_table(
    values='Profit_Margin',
    index='Meal',
    columns='Academic_Period'
).round(2)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, cbar_kws={'label': 'Profit Margin'})
plt.title('Profit Margin by Meal and Academic Period', fontsize=13, fontweight='bold')
plt.ylabel('')
plt.xlabel('')
plt.tight_layout()
plt.show()
print('Green = high profit margin   Red = low / negative margin')

In [ ]:
# Average profit margin per meal — bar chart
meal_avg = agg.groupby('Meal')['Profit_Margin'].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 4))
colors = ['seagreen' if v > meal_avg.mean() else 'salmon' for v in meal_avg.values]
meal_avg.plot(kind='bar', color=colors, edgecolor='white')
plt.axhline(meal_avg.mean(), color='black', linestyle='--',
            linewidth=1.5, label='Average')
plt.title('Average Profit Margin per Meal', fontsize=13, fontweight='bold')
plt.ylabel('Profit Margin')
plt.xlabel('')
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 6 — Scale the Features
Before feeding data into GMM, we normalise all features to the same scale.

> **Why?** Profit margin ranges from 0 to 1, but revenue is in millions of UGX. 
> Without scaling, revenue would dominate the clustering and the other features would be ignored.

In [ ]:
FEATURES = ['Profit_Margin', 'Waste_Ratio', 'Sell_Through', 'Revenue_Per_Sold']

X = agg[FEATURES].values.astype(float)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features used:', FEATURES)
print('Shape        :', X_scaled.shape)
print()
print('Before scaling — Profit_Margin range:', X[:, 0].min().round(3), 'to', X[:, 0].max().round(3))
print('After  scaling — Profit_Margin range:', X_scaled[:, 0].min().round(3), 'to', X_scaled[:, 0].max().round(3))

---
## Step 7 — Find the Best Number of Clusters
We try different numbers of clusters (2 to 6) and use the 
**Silhouette Score** to find the best one.

**Silhouette Score** measures how well each data point fits its cluster 
compared to neighbouring clusters.

- Score close to **+1** → clusters are well separated ✅
- Score close to **0** → clusters overlap too much ⚠️
- Score close to **-1** → points are in the wrong cluster ❌

> We pick the number of clusters with the **highest silhouette score**.

In [ ]:
print('Testing different numbers of clusters...')
print()

sil_scores = {}

for n in range(2, 7):
    gmm    = GaussianMixture(n_components=n, covariance_type='full',
                              random_state=42, max_iter=300)
    labels = gmm.fit_predict(X_scaled)
    score  = silhouette_score(X_scaled, labels)
    sil_scores[n] = score
    bar    = '█' * int(score * 40)
    print(f'  n={n} clusters  →  Silhouette Score = {score:.4f}  {bar}')

best_n = max(sil_scores, key=sil_scores.get)
print()
print(f'✅ Best number of clusters: {best_n}  (score = {sil_scores[best_n]:.4f})')

In [ ]:
# Plot silhouette scores
plt.figure(figsize=(8, 4))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()),
         marker='o', linewidth=2.5, color='steelblue', markersize=8)
plt.axvline(best_n, color='red', linestyle='--', linewidth=2,
            label=f'Best = {best_n} clusters')
plt.title('Silhouette Score vs Number of Clusters', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score (higher = better)')
plt.xticks(range(2, 7))
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Train the GMM
Now we train the GMM with the best number of clusters we just found.

The GMM will:
1. Assign each meal-period combination to a profitability cluster
2. Give a **probability** of belonging to each cluster

In [ ]:
# Train GMM with the best number of clusters
gmm = GaussianMixture(
    n_components    = best_n,
    covariance_type = 'full',   # each cluster can have its own shape
    random_state    = 42,
    max_iter        = 300
)
gmm.fit(X_scaled)

# Hard assignment — which cluster does each row belong to most?
agg['Cluster'] = gmm.predict(X_scaled)

# Soft assignment — probability of belonging to each cluster
proba = gmm.predict_proba(X_scaled)
for i in range(best_n):
    agg[f'Prob_C{i}'] = proba[:, i].round(3)

print(f'✅ GMM trained  |  Converged: {gmm.converged_}')
print()
print('Records per cluster:')
print(agg['Cluster'].value_counts().sort_index())

---
## Step 9 — Label Clusters (LOW / MEDIUM / HIGH)
We rank the clusters by average profit margin and assign labels.

The cluster with the highest average profit margin = **HIGH**,
the lowest = **LOW**.

In [ ]:
# Profile each cluster — what does the average record in each cluster look like?
profile = (
    agg.groupby('Cluster')
    .agg(
        Avg_Profit_Margin = ('Profit_Margin', 'mean'),
        Avg_Waste_Ratio   = ('Waste_Ratio',   'mean'),
        Avg_Sell_Through  = ('Sell_Through',  'mean'),
        Avg_Gross_Profit  = ('Avg_Gross_Profit', 'mean'),
        Count             = ('Meal', 'count')
    )
    .reset_index()
    .sort_values('Avg_Profit_Margin', ascending=False)
    .reset_index(drop=True)
)

# Assign labels: highest profit = HIGH, lowest = LOW
if len(profile) == 2:
    profit_labels = ['HIGH', 'LOW']
elif len(profile) == 3:
    profit_labels = ['HIGH', 'MEDIUM', 'LOW']
else:
    # For more clusters, spread them across tiers
    third = len(profile) / 3
    profit_labels = []
    for rank in range(len(profile)):
        if rank < third:
            profit_labels.append('HIGH')
        elif rank < 2 * third:
            profit_labels.append('MEDIUM')
        else:
            profit_labels.append('LOW')

profile['Profitability'] = profit_labels

# Map profitability label back to main table
label_map            = dict(zip(profile['Cluster'], profile['Profitability']))
agg['Profitability'] = agg['Cluster'].map(label_map)

# Also rename probability columns to be readable
for _, row in profile.iterrows():
    old = f'Prob_C{int(row["Cluster"])}'
    new = f'Prob_{row["Profitability"]}'
    if old in agg.columns and new not in agg.columns:
        agg.rename(columns={old: new}, inplace=True)

# Print cluster summary
icons = {'HIGH':'🟢', 'MEDIUM':'🟡', 'LOW':'🔴'}
print('Cluster Summary')
print('─' * 55)
for _, row in profile.iterrows():
    icon = icons.get(row['Profitability'], '')
    print(f"  {icon} {row['Profitability']:<8}  "
          f"Profit Margin: {row['Avg_Profit_Margin']*100:.1f}%   "
          f"Waste Ratio: {row['Avg_Waste_Ratio']*100:.1f}%   "
          f"Sell-Through: {row['Avg_Sell_Through']*100:.1f}%   "
          f"n={int(row['Count'])}")

---
## Step 10 — Visualise the Clusters
We use PCA to reduce our 4 features to 2 dimensions so we can 
plot them on a simple scatter chart.

> PCA is only used here for visualisation — the GMM was trained on all 4 features.

In [ ]:
# Reduce to 2D for plotting
pca   = PCA(n_components=2, random_state=42)
X_2d  = pca.fit_transform(X_scaled)

agg['PC1'] = X_2d[:, 0]
agg['PC2'] = X_2d[:, 1]

variance = pca.explained_variance_ratio_ * 100
print(f'Variance captured: PC1={variance[0]:.1f}%  PC2={variance[1]:.1f}%  '
      f'Total={variance.sum():.1f}%')

In [ ]:
palette = {'HIGH':'seagreen', 'MEDIUM':'steelblue', 'LOW':'salmon'}

plt.figure(figsize=(10, 6))

for prof, color in palette.items():
    subset = agg[agg['Profitability'] == prof]
    if len(subset) == 0:
        continue
    plt.scatter(
        subset['PC1'], subset['PC2'],
        c=color, label=f'{prof} profit',
        s=90, alpha=0.85, edgecolors='white', linewidths=0.5
    )
    # Label each point with meal name
    for _, row in subset.iterrows():
        plt.annotate(
            f"{row['Meal'][:10]}\n{row['Academic_Period'][:4]}",
            (row['PC1'], row['PC2']),
            fontsize=6.5, alpha=0.8,
            xytext=(4, 4), textcoords='offset points'
        )

plt.title('GMM Profitability Clusters — Meal × Academic Period',
          fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({variance[0]:.1f}% of variance)')
plt.ylabel(f'PC2 ({variance[1]:.1f}% of variance)')
plt.legend(fontsize=10, title='Profitability')
plt.tight_layout()
plt.show()
print('Each dot = one meal in one academic period.')
print('Dots close together = similar profitability patterns.')

---
## Step 11 — Probabilities per Meal and Academic Period
This is the key GMM output — instead of just a label, 
each meal gets a **probability** for each profitability cluster.

**How to read this:**
- `Prob_HIGH = 0.90` → the model is 90% confident this is a high profit meal
- `Prob_HIGH = 0.55` → borderline — could go either way, needs attention
- `Prob_LOW  = 0.85` → this meal is consistently unprofitable

In [ ]:
# Build clean probability table
prob_cols = [c for c in agg.columns if c.startswith('Prob_')]

prob_table = agg[['Meal', 'Academic_Period', 'Profitability'] + prob_cols].copy()
prob_table = prob_table.sort_values(['Meal', 'Academic_Period']).reset_index(drop=True)

print('Soft cluster probabilities per meal × academic period:')
print()
prob_table

In [ ]:
# Heatmap: probability of HIGH profit per meal per period
if 'Prob_HIGH' in agg.columns:
    pivot_prob = agg.pivot_table(
        values='Prob_HIGH',
        index='Meal',
        columns='Academic_Period'
    ).round(2)

    plt.figure(figsize=(10, 5))
    sns.heatmap(pivot_prob, annot=True, fmt='.2f',
                cmap='RdYlGn', vmin=0, vmax=1,
                linewidths=0.5,
                cbar_kws={'label': 'Probability of HIGH Profit'})
    plt.title('How Likely is Each Meal to be High Profit?\nby Academic Period',
              fontsize=13, fontweight='bold')
    plt.ylabel('')
    plt.xlabel('')
    plt.tight_layout()
    plt.show()
    print('1.00 = definitely high profit   0.00 = definitely not high profit')
    print('Values between 0.40–0.65 = borderline meals, monitor closely')

---
## Step 12 — Profitability by Academic Period
Now we directly answer the question: 
*how does each meal's profitability change across academic periods?*

In [ ]:
# Bar chart: cluster distribution per academic period
period_counts = pd.crosstab(agg['Academic_Period'], agg['Profitability'])
col_order     = [c for c in ['HIGH','MEDIUM','LOW'] if c in period_counts.columns]
period_counts = period_counts[col_order]

period_counts.plot(
    kind='bar', figsize=(10, 5),
    color=['seagreen','steelblue','salmon'][:len(col_order)],
    edgecolor='white', rot=15
)
plt.title('How Many Meals Fall into Each Profitability Tier\nby Academic Period',
          fontsize=13, fontweight='bold')
plt.ylabel('Number of Meals')
plt.xlabel('')
plt.legend(title='Profitability')
plt.tight_layout()
plt.show()

In [ ]:
# Clean summary table: meal profitability per academic period
summary = agg[['Meal','Academic_Period','Profitability',
               'Avg_Gross_Profit','Avg_Revenue',
               'Profit_Margin','Waste_Ratio']].copy()

summary['Profit_Margin'] = (summary['Profit_Margin'] * 100).round(1)
summary['Waste_Ratio']   = (summary['Waste_Ratio']   * 100).round(1)
summary['Avg_Gross_Profit'] = summary['Avg_Gross_Profit'].round(0)
summary['Avg_Revenue']      = summary['Avg_Revenue'].round(0)

summary.columns = ['Meal','Academic Period','Profitability',
                   'Avg Gross Profit (UGX)','Avg Revenue (UGX)',
                   'Profit Margin %','Waste %']

summary = summary.sort_values(['Meal','Academic Period']).reset_index(drop=True)
print('Profitability per meal × academic period:')
print()
summary

---
## Step 13 — Manager Report
A simple printout the cafeteria manager can read and act on.

In [ ]:
def profitability_report(cafeteria_id='BBG'):
    icons  = {'HIGH':'🟢', 'MEDIUM':'🟡', 'LOW':'🔴'}
    border = '=' * 65
    data   = agg[agg['Cafeteria_ID'] == cafeteria_id]

    if data.empty:
        print('No data for cafeteria:', cafeteria_id)
        return

    print(f'\n{border}')
    print(f'  MEAL PROFITABILITY REPORT  —  {cafeteria_id}')
    print(f'{border}')

    for meal in sorted(data['Meal'].unique()):
        rows = data[data['Meal'] == meal].sort_values('Academic_Period')
        print(f'\n  {meal}')
        print(f'  {"-"*60}')
        print(f'  {"Period":<22} {"Label":>10}  {"Margin":>8}  {"Confidence":>12}')
        print(f'  {"-"*22} {"-"*10}  {"-"*8}  {"-"*12}')

        for _, row in rows.iterrows():
            prof     = row['Profitability']
            icon     = icons.get(prof, '')
            margin   = row['Profit_Margin'] * 100
            prob_col = f'Prob_{prof}'
            conf     = row[prob_col] * 100 if prob_col in row.index else 0
            flag     = '  ← borderline' if 40 < conf < 65 else ''
            print(f"  {row['Academic_Period']:<22} {icon} {prof:<8}  "
                  f"{margin:>6.1f}%  {conf:>10.1f}%{flag}")

    print(f'\n{border}')
    print('  🟢 HIGH   → keep and promote')
    print('  🟡 MEDIUM → optimise pricing or ingredients')
    print('  🔴 LOW    → consider repricing or removing')
    print('  ← borderline → GMM is uncertain, monitor this closely')
    print(f'{border}\n')


profitability_report('BBG')

---
## Step 14 — Save Results

In [ ]:
save_cols = (['Cafeteria_ID','Meal','Academic_Period','Profitability',
               'Profit_Margin','Avg_Gross_Profit','Avg_Revenue',
               'Waste_Ratio','Sell_Through']
              + [c for c in agg.columns if c.startswith('Prob_')])
save_cols = [c for c in save_cols if c in agg.columns]

agg[save_cols].to_csv('meal_profitability_results.csv', index=False)
print('✅ Saved: meal_profitability_results.csv')

---
## ✅ Summary

Here is everything we did in simple terms:

1. **Loaded** the cafeteria dataset
2. **Computed** four profitability features per record
3. **Aggregated** records into one row per meal × academic period
4. **Scaled** features so they are all on the same level
5. **Used Silhouette Score** to find the best number of clusters
6. **Trained GMM** — it found natural profitability groups in the data
7. **Labelled** clusters as LOW / MEDIUM / HIGH profitability
8. **Got probabilities** — how confident is the model about each meal?
9. **Saw how profitability shifts** across academic periods
10. **Printed a manager report** with actionable recommendations

### The one thing GMM gives you that K-Means cannot

A **confidence score** per meal. When the model says a meal is HIGH profit 
with 90% confidence — you can trust that. When it says HIGH with 52% confidence — 
that meal is borderline and needs closer attention.